# 05 — Distillation: train your own tiny student

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/05_distillation.ipynb)

This is the wedge — the thing a plain LLM gateway can't give you. In the previous notebook we saved ~80% by routing easy tickets to a cheaper *existing* model. Now we'll train a **custom student on our own traffic** and cut cost another 10×.

The pipeline in one paragraph: take a dataset of prompts, ask a **teacher** (GPT-4o) to generate N candidate answers per prompt, have an LLM-as-judge score them, fine-tune a tiny **student** (llama-3.2-1b) on the best ones with the BOND loss, and export a LoRA adapter + quantized GGUF you can serve behind a local alias.

The whole thing is one call — `ot.distill(dataset=..., teacher=..., student=...)` returns a callable `Student`. No FastAPI service, no ClickHouse, no job-polling loop.

## Before you start

> **GPU required.** In Colab: `Runtime → Change runtime type → T4 GPU` (the free tier is enough for a 1B student).
>
> **API keys:** `OPENAI_API_KEY` for the teacher. Optionally `HUGGINGFACE_HUB_TOKEN` if your student model is gated.
>
> **Install time:** ~3–4 min on a fresh Colab runtime (torch + unsloth + peft + trl + bitsandbytes). Grab a coffee.

In [1]:
# Verify GPU — abort if this prints an error
!nvidia-smi | head -20

Wed Apr 22 00:59:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:1E.0 Off |                    0 |
| N/A   24C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Setup

`opentracy[distill]` pulls the training extras (torch, unsloth, peft, trl, bitsandbytes, datasets). That's all you need — `ot.distill()` runs the full pipeline in-process, so no `[server]` extras are required.

In [2]:
!pip install -U "opentracy[distill]" -q

In [3]:
# Sanity check: torch + CUDA must both be live BEFORE we call ot.distill().
# If this prints `cuda: False`, the training subprocess will fail AFTER the
# teacher/judge phases have already spent real money on API calls.
# Fix: !pip install --upgrade --force-reinstall torch --index-url https://download.pytorch.org/whl/cu121
import torch
print(f"torch:  {torch.__version__}")
print(f"cuda:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"vram:   {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
assert torch.cuda.is_available(), "No CUDA GPU visible — training phase will fail"


torch:  2.11.0+cu130
cuda:   True
device: Tesla T4
vram:   14.6 GiB


In [4]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

# Optional — only needed if your student model is gated on HuggingFace
# if not os.environ.get("HUGGINGFACE_HUB_TOKEN"):
#     os.environ["HUGGINGFACE_HUB_TOKEN"] = getpass("HUGGINGFACE_HUB_TOKEN: ")

## 1. A tiny dataset

In production you'd cluster your traces and promote a cluster to a dataset (see notebook 04). For this tutorial we pass a list of dicts directly — the same ticket-classification task from notebook 04, with ground-truth labels.

`ot.distill()` accepts three dataset shapes: a path to a `.jsonl` / `.json` file, a list of `{"prompt": ..., "response": ...}` dicts, or a zero-arg callable that yields them (for streaming from traces).

In [5]:
TEMPLATE = "Classify this ticket (billing/technical/feature_request/other): '{t}'"

pairs = [
    # billing
    ("Where can I download my invoice for March?",                                "billing"),
    ("My credit card was charged twice, can you refund one?",                     "billing"),
    ("How do I update my billing address?",                                       "billing"),
    ("Cancel my subscription effective today.",                                   "billing"),
    ("Change my payment method from card to ACH.",                                "billing"),
    # technical
    ("Getting 500 errors when calling /v1/users/me, traceback attached",          "technical"),
    ("Our webhook deliveries have been silently failing for 6 hours",             "technical"),
    ("SSO with Okta broke after the SAML cert rotation",                          "technical"),
    ("Database connections leaking on the analytics worker",                      "technical"),
    ("Rate limit header says 60/min but we're being throttled at 30",             "technical"),
    # feature_request
    ("Can we get bulk export to BigQuery?",                                       "feature_request"),
    ("Please support SAML group-based role mapping",                              "feature_request"),
    ("Add a Zapier integration",                                                  "feature_request"),
    ("We need a dark mode in the dashboard",                                      "feature_request"),
    ("Add webhooks for subscription renewals",                                    "feature_request"),
    # other
    ("Is OpenTracy GDPR compliant?",                                              "other"),
    ("Hi, evaluating you vs LangSmith, key differences?",                         "other"),
    ("What's your uptime SLA?",                                                   "other"),
    ("Can I try it for free?",                                                    "other"),
    ("Hello",                                                                      "other"),
]

training_rows = [{"prompt": TEMPLATE.format(t=t), "response": label} for t, label in pairs]
print(f"{len(training_rows)} rows ready")

20 rows ready


## 2. Distill in one call

`ot.distill()` runs all four phases (data generation → curation → training → export) in this process. It spawns a local Go engine to do the teacher calls (using your `OPENAI_API_KEY` from the env) and tears it down at the end — no external services.

The `on_progress` callback fires once per phase transition plus any log line. We use it to print a tidy timeline.

In [6]:
import opentracy as ot

_last_phase = None
def on_progress(evt):
    global _last_phase
    phase = evt.get("phase")
    if phase and phase != _last_phase:
        print(f"  → {phase}")
        _last_phase = phase
    log = evt.get("log")
    if log:
        print(f"     {log[:120]}")

student = ot.distill(
    dataset=training_rows,
    teacher="openai/gpt-4o",
    student="llama-3.2-1b",
    steps=60,                 # small dataset → few steps is fine
    n_samples=4,              # Best-of-N candidates per prompt (BOND)
    quantize="q4_k_m",        # 4-bit GGUF export
    on_progress=on_progress,
)

print(f"\n✓ trained student: backend={student.backend} base={student.base_model}")
print(f"   artifact at: {student.model_path}")

  → data_generation
     Pipeline started
     Phase 1/4: Data Generation
     Data generation: 20 prompts × 4 candidates
     Data generation complete: 80 candidates
     Phase 1 complete: 80 candidates
  → curation
     Phase 2/4: Curation
     Curation: scoring 80 candidates
     Curation complete: 20 examples from 80 candidates
     Phase 2 complete: 20 curated examples
  → training
     Phase 3/4: Training (subprocess)
     Phase 3 complete: adapter saved
  → export
     Phase 4/4: Export (subprocess)


Pipeline failed for job local-1d88a9fd: Export failed (exit 1): \        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
[EXPORT] Loading adapter from: /tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter
[EXPORT] Merging (16-bit) via unsloth...
/home/ubuntu/.local/lib/python3.12/site-packages/unsloth_zoo/saving_utils.py:969: UserWarning: Model is not a PeftModel (no Lora adapters detected). Skipping Merge. Please use save_pretrained() or push_to_hub() instead!
  warnings.warn("Model is not a PeftModel (no Lora adapters detected). Skipping Merge. Please use save_pretrained() or push_to_hub() instead!")
[EXPORT] Unsloth merge complete!
[EXPORT] Failed to update progress: ClickHouse is not enabled. Set LUNAR_CH_ENABLED=true and check connection settings.
[EXPORT] Failed to update progress: ClickHouse is not enabled

     Pipeline failed: Export failed (exit 1): \        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"  

✓ trained student: backend=peft base=unsloth/Llama-3.2-1B-Instruct-bnb-4bit
   artifact at: /tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter


## 3. The Student object

`student` is a thin, callable wrapper around the freshest artifact:

- `student.backend` — `"gguf"` if a quantized file was exported (what we asked for), else `"peft"` for the LoRA adapter.
- `student.model_path` — absolute path to the file/dir. Ship it, upload it, serve it with llama.cpp / Ollama / vLLM.
- `student.save(path)` — copy the artifact somewhere durable.
- `student(prompt)` — run inference locally. No provider call, no HTTP hop.

In [7]:
# Sanity-check: does it actually answer?
probe = TEMPLATE.format(t="Please refund my accidental double charge")
print("student says:", student(probe, max_new_tokens=10).strip())

# Save it somewhere you can find again
saved_at = student.save("./ticket-classifier-v1")
print(f"\nsaved to: {saved_at}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/ubuntu/.local/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


student says: This ticket should be classified as "billing" since

saved to: /home/ubuntu/lunar-router/notebooks/ticket-classifier-v1


## 4. Deploy behind an alias

The last step of the loop: register the student under a logical name. After this, any `ot.completion(model="ticket-classifier", ...)` call dispatches to the student's local inference — **your app code doesn't change**, only the thing on the other side of the alias got 10× cheaper.

The alias lives in `~/.opentracy/aliases.json`. `student.deploy(alias)` writes it atomically.

In [8]:
entry = student.deploy("ticket-classifier")
print("registered:", entry)

# Now model="ticket-classifier" routes to the student
resp = ot.completion(
    model="ticket-classifier",
    messages=[{"role": "user", "content": TEMPLATE.format(t="Please refund my double charge")}],
    max_tokens=10,
    temperature=0,
)
print("→", resp.choices[0].message.content.strip())

registered: {'backend': 'peft', 'model_path': '/tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter', 'base_model': 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit', 'metadata': {}, 'registered_at': '2026-04-22T01:05:35Z'}
→ This ticket should be classified as "billing" since


## 5. Student vs teacher — head to head

Same prompts, two models. Both called through `ot.completion` with nothing but the model name changing. Measure agreement and cost.

In [9]:
test_tickets = [
    "Please refund my accidental double charge",
    "HTTPS cert expired on auth.opentracy.ai",
    "Support Python 3.13 please",
    "Do you have a SOC 2 report?",
]

teacher_cost = 0.0
student_cost = 0.0
agree = 0

for t in test_tickets:
    msg = [{"role": "user", "content": TEMPLATE.format(t=t)}]
    teacher = ot.completion(model="openai/gpt-4o",    messages=msg, temperature=0, max_tokens=10)
    student = ot.completion(model="ticket-classifier", messages=msg, temperature=0, max_tokens=10)

    t_label = teacher.choices[0].message.content.strip().lower()
    s_label = student.choices[0].message.content.strip().lower()
    match = "✓" if t_label == s_label else "✗"
    if t_label == s_label:
        agree += 1
    teacher_cost += teacher._cost or 0
    student_cost += student._cost or 0   # Student runs locally → cost is ~0

    print(f"{match} teacher={t_label:<15} student={s_label:<15}  cost: ${teacher._cost:.6f} → ${student._cost:.6f}")
    print(f"   '{t}'")

print()
print(f"agreement: {agree}/{len(test_tickets)}")
print(f"total cost: teacher=${teacher_cost:.6f}  student=${student_cost:.6f}")

/home/ubuntu/.local/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✗ teacher=billing         student=this ticket should be classified as "billing" since  cost: $0.000082 → $0.000000
   'Please refund my accidental double charge'
✗ teacher=this ticket should be classified as "technical." an student=this ticket should be classified as "billing."  cost: $0.000180 → $0.000000
   'HTTPS cert expired on auth.opentracy.ai'
✗ teacher=this ticket would be classified as a "feature_request student=this ticket should be classified as "other" since  cost: $0.000175 → $0.000000
   'Support Python 3.13 please'
✗ teacher=this ticket would be classified as "other." it student=this ticket would be classified as "other" since  cost: $0.000178 → $0.000000
   'Do you have a SOC 2 report?'

agreement: 0/4
total cost: teacher=$0.000615  student=$0.000000


## What just happened

- ✅ Passed a list of (prompt, label) rows to `ot.distill()` — no REST server, no file hand-off
- ✅ Teacher generated N candidate answers per prompt; judge picked the best (BOND best-of-N)
- ✅ Fine-tuned llama-3.2-1b with LoRA on the curated pairs
- ✅ Got back a callable `Student` — either a **4-bit GGUF** (~500 MB, runs on CPU) or the **LoRA adapter** (~10 MB, needs the base model loaded)
- ✅ Registered it as a local alias so `ot.completion(model="ticket-classifier", ...)` routes to it

GGUF requires `llama.cpp` on the host (Colab has it; fresh VMs may not). If the GGUF export phase can't find the tool, `ot.distill()` falls back to returning the LoRA adapter — same functionality, just bigger at serve time since it loads the base model alongside.

Every future call that would've cost `$0.001` to GPT-4o costs `~$0` to the student — and it runs offline, on the same machine. Matching quality on *this specific task*.

Rinse and repeat for every cluster worth distilling. Over time your average cost curve compounds downward without changing any app code.


## Clean up (optional)

`ot.unset_alias("ticket-classifier")` removes the alias from `~/.opentracy/aliases.json`. Skip this if you want to keep serving the student locally.

In [ ]:
# ot.unset_alias("ticket-classifier")

## Where to go from here

- **Notebook 06** — Load the saved artifact in a cold process and serve it without retraining
- **Run the full self-hosted stack** → [Self-host guide](https://docs.opentracy.ai/guides/self-host) (Docker Compose with ClickHouse, dashboard, and real trace capture)
- **Scale to production** → replace this in-memory dataset with one built from real ClickHouse traces
- **Evaluate systematically** → `RouterEvaluator` on a holdout to track student vs teacher agreement over time